In [1]:
import pandas as pd
import numpy as np

In [ ]:
cd C:\Github_repository\Metabolomics_Tutorial\Data

In [3]:
STYPE = 'fecal_neg'
WEEK = '2w'

In [4]:
df1 = pd.read_excel('dataset_filtered_scaled_merged.xlsx', sheet_name=f'{STYPE}_{WEEK}', skiprows=[1])  ## 여기에 원본 Compounds 파일 넣고
df1 = df1.rename(columns={'Label': 'FeatureID'})
# 1) 첫 컬럼명을 FeatureID로
df1 = df1.rename(columns={df1.columns[0]: "FeatureID"})
# 2) 2행(=그룹 라벨 행) 제거: FeatureID 값이 'Group' 또는 'Label'인 행
mask = df1["FeatureID"].astype(str).str.strip().str.lower().isin(["group", "label"])
df1 = df1.loc[~mask].reset_index(drop=True)

df2 = pd.read_excel('stats_merged.xlsx', sheet_name=f'{STYPE}_{WEEK}')  ## 여기엔 FDR로 필터링한 파일 넣고

## 매칭용 이름 정리
df1["Name_key"] = df1["FeatureID"].astype(str).str.strip()
df2["Name_key"] = df2["FeatureID"].astype(str).str.strip()

df1

,FeatureID,C01_2w,C02_2w,C03_2w,C04_2w,C05_2w,C06_2w,C07_2w,C08_2w,C09_2w,...,V02_2w,V03_2w,V04_2w,V05_2w,V06_2w,V07_2w,V08_2w,V09_2w,V10_2w,Name_key
0,296.61697-8.708,-4.572774,0.223411,0.657662,-0.098263,0.496152,0.948159,0.618992,0.542554,1.009734,...,1.106246,1.038808,1.110776,-4.572774,1.003630,1.012000,0.896026,0.766784,0.948976,296.61697-8.708
1,487.23704-10.048,0.067086,0.453396,0.402793,0.660216,0.991129,0.245269,0.808271,-1.357107,0.543340,...,-0.002437,-0.973854,-1.232324,1.037276,-0.680447,-0.506565,-1.041567,-1.077987,-1.641813,487.23704-10.048
2,134.89469-16.398,1.110949,-0.553325,-1.168239,0.095419,-0.624860,0.859972,-0.493933,0.482891,-0.482458,...,0.009938,-0.411548,0.119807,0.034094,0.289589,0.488096,0.136336,0.034401,0.110369,134.89469-16.398
3,421.22647-16.397,-0.208549,-0.280528,-1.001226,0.724640,0.021564,0.014339,-0.897659,-0.042201,-0.645183,...,-0.066963,-0.345408,0.776809,-0.229924,0.056632,0.812477,0.851845,0.826798,-0.304622,421.22647-16.397
4,296.61713-13.231,-2.462796,-2.462796,-2.462796,-2.462796,-2.462796,1.835018,1.406076,-2.462796,1.875993,...,2.124917,1.984142,1.968177,-2.462796,2.026062,2.017696,1.789133,1.620294,1.880161,296.61713-13.231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8564,520.83183-0.955,0.156101,0.534191,-1.894465,0.518628,-1.894465,-1.894465,-1.894465,-1.894465,-1.894465,...,1.754321,0.106558,1.735748,1.456689,0.573400,1.517752,1.762983,0.845962,1.257555,520.83183-0.955
8565,356.91342-0.925,0.725600,-0.122333,0.264382,-0.044398,-1.616397,0.118876,0.522011,0.205893,0.483042,...,0.921243,-0.469283,-0.615441,0.136369,-0.020823,-0.022763,-0.219156,-0.279917,0.444356,356.91342-0.925
8566,127.00133-1,0.405881,0.729615,0.889764,0.451989,0.711235,-2.675664,-2.675664,0.989545,-2.675664,...,0.890606,-2.675664,0.691364,0.114295,0.462780,0.280555,0.240864,1.081080,0.930098,127.00133-1
8567,443.23-9.348,-1.493018,-0.778087,-0.536311,-0.465949,0.562218,0.266093,-0.747465,-0.525304,0.910013,...,0.865122,-0.136649,1.155890,-0.609713,-0.031706,0.029071,0.594503,1.098527,0.466718,443.23-9.348


In [5]:
# df2에서 FeatureID와 Name 같이 가져오기 (Name 없으면 에러)
if "Name" not in df2.columns:
    raise ValueError("df2에 Name 컬럼이 없습니다. df2에서 Name 컬럼명을 확인하세요.")
if "q-val_BKY" not in df2.columns:
    raise ValueError(f"df2에 q-val_BKY 컬럼이 없습니다. 현재 컬럼: {list(df2.columns)}")

# 매칭 키 정리(공백 제거)
df1["FeatureID"] = df1["FeatureID"].astype(str).str.strip()
df2["FeatureID"] = df2["FeatureID"].astype(str).str.strip()

# q-val_BKY 숫자화 후 유의한 FeatureID만 남기기
df2["q-val_BKY"] = pd.to_numeric(df2["q-val_BKY"], errors="coerce")

df2_sig = (
    df2.loc[df2["q-val_BKY"] < 0.05, ["FeatureID", "Name"]]
      .dropna(subset=["FeatureID"])
      .drop_duplicates(subset=["FeatureID"], keep="first")
)

# df1에 df2_sig만 inner join (== 유의한 FeatureID만 매칭)
df1_overlap = pd.merge(
    df1,
    df2_sig,
    on="FeatureID",
    how="inner"
)

print("df1:", df1.shape, "df2_sig:", df2_sig.shape, "overlap:", df1_overlap.shape)
df1_overlap

df1: (8569, 23) df2_sig: (7281, 2) overlap: (7281, 24)


,FeatureID,C01_2w,C02_2w,C03_2w,C04_2w,C05_2w,C06_2w,C07_2w,C08_2w,C09_2w,...,V03_2w,V04_2w,V05_2w,V06_2w,V07_2w,V08_2w,V09_2w,V10_2w,Name_key,Name
0,296.61697-8.708,-4.572774,0.223411,0.657662,-0.098263,0.496152,0.948159,0.618992,0.542554,1.009734,...,1.038808,1.110776,-4.572774,1.003630,1.012000,0.896026,0.766784,0.948976,296.61697-8.708,NaN
1,487.23704-10.048,0.067086,0.453396,0.402793,0.660216,0.991129,0.245269,0.808271,-1.357107,0.543340,...,-0.973854,-1.232324,1.037276,-0.680447,-0.506565,-1.041567,-1.077987,-1.641813,487.23704-10.048,NaN
2,421.22647-16.397,-0.208549,-0.280528,-1.001226,0.724640,0.021564,0.014339,-0.897659,-0.042201,-0.645183,...,-0.345408,0.776809,-0.229924,0.056632,0.812477,0.851845,0.826798,-0.304622,421.22647-16.397,"1,4-Bis(2-ethylhexyl) sulfosuccinate"
3,296.61713-13.231,-2.462796,-2.462796,-2.462796,-2.462796,-2.462796,1.835018,1.406076,-2.462796,1.875993,...,1.984142,1.968177,-2.462796,2.026062,2.017696,1.789133,1.620294,1.880161,296.61713-13.231,NaN
4,197.8081-1.761,1.956116,1.923118,2.182810,2.083548,1.352929,2.251600,1.961678,2.226267,1.761642,...,-2.214149,-2.214149,2.196438,-2.214149,-2.214149,-2.214149,-2.214149,-2.214149,197.8081-1.761,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7276,291.73107-0.859,0.843714,0.868264,0.929613,0.500311,0.917359,0.330793,1.070613,1.033744,0.954668,...,0.350412,-3.008888,0.678279,-0.112341,-0.546428,-3.008888,-0.270721,0.694172,291.73107-0.859,NaN
7277,555.02091-1.006,0.293503,0.625360,0.158100,0.340466,0.606416,-2.919896,-2.919896,0.489472,-2.919896,...,0.311062,-0.325895,0.000827,1.237861,0.709862,1.213525,1.126579,1.282125,555.02091-1.006,NaN
7278,318.8747-0.972,0.421251,0.990746,-2.623551,-0.051878,1.225490,-2.623551,0.087669,-2.623551,-2.623551,...,0.269020,0.498428,-0.237364,1.119238,1.078428,0.580941,0.613916,0.551141,318.8747-0.972,NaN
7279,454.84771-0.969,0.289999,0.338911,1.171263,-0.210489,0.347164,1.223844,-0.489573,-2.904141,-2.904141,...,0.978856,1.089292,1.372921,0.004102,0.201769,0.458925,0.675809,0.427401,454.84771-0.969,NaN


In [6]:
# 4) 주차 컬럼 선택 + C/V 샘플만
wk = f'_{WEEK}'
group_area_cols = [c for c in df1_overlap.columns if wk in str(c)]
group_area_cols = [c for c in group_area_cols if str(c).strip().startswith(("C","V"))]
print("selected sample cols:", len(group_area_cols))
group_area_cols

selected sample cols: 21


['C01_2w',
 'C02_2w',
 'C03_2w',
 'C04_2w',
 'C05_2w',
 'C06_2w',
 'C07_2w',
 'C08_2w',
 'C09_2w',
 'C10_2w',
 'C11_2w',
 'V01_2w',
 'V02_2w',
 'V03_2w',
 'V04_2w',
 'V05_2w',
 'V06_2w',
 'V07_2w',
 'V08_2w',
 'V09_2w',
 'V10_2w']

In [7]:
# (추가) GroupArea 컬럼들 숫자화
df1_overlap[group_area_cols] = (
    df1_overlap[group_area_cols]
      .replace(r'^\s*$', np.nan, regex=True)     # 빈칸/공백 -> NaN
      .apply(lambda s: pd.to_numeric(s, errors='coerce'))
)

# 5) Name 중복 제거: (선택 주차) 평균이 큰 1개만
df1_overlap["GroupArea_mean"] = df1_overlap[group_area_cols].mean(axis=1, skipna=True)

df1_overlap = df1_overlap.sort_values(by=["Name", "GroupArea_mean"], ascending=[True, False])
df1_overlap_unique = df1_overlap.drop_duplicates(subset=["Name"], keep="first")

df1_overlap_unique

,FeatureID,C01_2w,C02_2w,C03_2w,C04_2w,C05_2w,C06_2w,C07_2w,C08_2w,C09_2w,...,V04_2w,V05_2w,V06_2w,V07_2w,V08_2w,V09_2w,V10_2w,Name_key,Name,GroupArea_mean
5203,451.12527-6.276,-0.541214,-0.162147,0.733899,0.396673,0.724164,0.292201,0.315721,0.577733,0.815524,...,-0.728592,0.598371,-0.831722,-0.470453,-0.956577,-0.404778,0.085645,451.12527-6.276,(+)-Catechin-7-O-glucoside,-2.149735e-15
65,293.17603-11.873,1.781424,1.305286,-2.092619,-2.092619,-2.092619,-2.092619,-2.092619,-2.092619,-2.092619,...,2.045422,1.241757,2.112647,2.219844,2.097427,-2.092619,2.176181,293.17603-11.873,(+)-[6]-Gingerol,1.416856e-15
6833,425.18219-11.004,-4.043619,-0.789239,-0.575115,-0.430061,-0.157027,-0.176574,-0.426084,-0.546895,-0.249212,...,0.758277,-0.382479,1.108722,1.100497,1.114623,1.130904,0.928751,425.18219-11.004,(+)-abscisic acid β-D-glucopyranosyl ester,-2.114711e-16
3861,188.09303-6.583,0.518113,-0.059633,-0.000150,0.148279,0.093143,0.174149,-0.082730,0.182425,0.098200,...,0.829558,0.192922,0.364117,-0.011534,0.401630,0.645419,0.539312,188.09303-6.583,(+)-castanospermine,-1.427430e-16
251,329.10332-9.316,0.785084,1.341589,0.986161,1.045167,1.000292,1.064841,1.030745,0.346368,1.272051,...,-1.313329,1.064054,-1.327913,-1.294552,-1.383440,-1.426581,-1.445913,329.10332-9.316,(+)-sesamin dicatechol,8.670313e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4531,221.0668-7.449,-1.698018,-1.698018,-1.698018,-1.698018,-1.698018,-1.698018,0.777734,-1.698018,-1.698018,...,1.900151,-1.698018,1.994699,2.028369,2.088049,2.110018,2.061940,221.0668-7.449,β-D-Ethyl glucuronide,2.918301e-15
809,180.06674-1.052,-1.573810,-0.720581,0.579979,-0.256715,-1.045909,-0.305610,-0.806348,-0.372038,-0.340994,...,0.892081,-0.393339,0.329789,0.791074,0.511125,0.908904,0.830213,180.06674-1.052,γ-Hydroxy-3-pyridinebutanoate,-3.700743e-16
1948,265.1196-5.897,1.056218,0.440152,1.447907,0.455353,0.746982,0.295844,-0.017269,1.697026,1.314824,...,-1.234382,0.371824,-1.523068,-0.719407,-0.781700,-0.387922,-0.180978,265.1196-5.897,γ-glutamyltyramine,-2.709473e-16
1068,274.14113-1.773,0.734452,0.795945,1.388045,0.768575,1.087602,0.652345,0.171664,1.365152,0.992343,...,-0.922430,0.670873,-1.529192,-1.292649,-0.945716,-1.077818,-0.593749,274.14113-1.773,ε-(γ-Glutamyl)-lysine,6.661338e-16


In [8]:
# 6) 저장은 Name으로(FeatureID는 매칭용으로만 사용)
result = df1_overlap_unique[["Name"] + group_area_cols].copy()

# 2행(Group) 추가 + 1열 헤더를 Label로 바꾸기(원하면)
out = result.rename(columns={"Name":"Label"}).copy()

def to_group(c):
    s = str(c).strip()
    if s.startswith("C"): return "Control"  #Edit
    if s.startswith("V"): return "VNAM"  #Edit
    return "NA"

group_row = ["Group"] + [to_group(c) for c in group_area_cols]
group_df = pd.DataFrame([group_row], columns=["Label"] + group_area_cols)

out2 = pd.concat([group_df, out], ignore_index=True)
out2.to_csv(f'MA_PW_{WEEK}_sig.csv', index=False, encoding="utf-8-sig")

print(out2.head(3))
print(out2.shape)

                        Label    C01_2w    C02_2w    C03_2w    C04_2w  \
0                       Group   Control   Control   Control   Control   
1  (+)-Catechin-7-O-glucoside -0.541214 -0.162147  0.733899  0.396673   
2            (+)-[6]-Gingerol  1.781424  1.305286 -2.092619 -2.092619   

     C05_2w    C06_2w    C07_2w    C08_2w    C09_2w  ...    V01_2w    V02_2w  \
0   Control   Control   Control   Control   Control  ...      VNAM      VNAM   
1  0.724164  0.292201  0.315721  0.577733  0.815524  ... -0.925093 -0.034507   
2 -2.092619 -2.092619 -2.092619 -2.092619 -2.092619  ...  1.944538  1.986057   

     V03_2w    V04_2w    V05_2w    V06_2w    V07_2w    V08_2w    V09_2w  \
0      VNAM      VNAM      VNAM      VNAM      VNAM      VNAM      VNAM   
1 -0.624487 -0.728592  0.598371 -0.831722 -0.470453 -0.956577 -0.404778   
2  2.015605  2.045422  1.241757  2.112647  2.219844  2.097427 -2.092619   

     V10_2w  
0      VNAM  
1  0.085645  
2  2.176181  

[3 rows x 22 columns]
(1233,